<a href="https://colab.research.google.com/github/raihansb29/complete-pandas-tutorial/blob/master/Pycaret_main_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Only enable critical logging (Optional)
import os
os.environ["PYCARET_CUSTOM_LOGGING_LEVEL"] = "CRITICAL"

In [1]:
def what_is_installed():
    from pycaret import show_versions
    show_versions()

try:
    what_is_installed()
except ModuleNotFoundError:
    !pip install pycaret
    what_is_installed()


System:
    python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
executable: /usr/bin/python3
   machine: Linux-6.1.123+-x86_64-with-glibc2.35

PyCaret required dependencies:
                 pip: 24.1.2
          setuptools: 75.2.0
             pycaret: 3.3.2
             IPython: 7.34.0
          ipywidgets: 7.7.1
                tqdm: 4.67.1
               numpy: 1.26.4
              pandas: 2.1.4
              jinja2: 3.1.6
               scipy: 1.11.4
              joblib: 1.3.2
             sklearn: 1.4.2
                pyod: 2.0.5
            imblearn: 0.13.0
   category_encoders: 2.7.0
            lightgbm: 4.6.0
               numba: 0.60.0
            requests: 2.32.3
          matplotlib: 3.7.5
          scikitplot: 0.3.7
         yellowbrick: 1.5
              plotly: 5.24.1
    plotly-resampler: Not installed
             kaleido: 1.0.0
           schemdraw: 0.15
         statsmodels: 0.14.5
              sktime: 0.26.0
               tbats: 1.1.3
            pmdar

In [2]:
import time
import numpy as np
import pandas as pd

from pycaret.datasets import get_data
from pycaret.time_series import TSForecastingExperiment


In [3]:
# We want to forecast the next 12 months of data and we will use 3 fold cross-validation to test the models.
fh = 12 # or alternately fh = np.arange(1,13)
fold = 3

In [4]:
# Global Figure Settings for notebook ----
# Depending on whether you are using jupyter notebook, jupyter lab, Google Colab, you may have to set the renderer appropriately
# NOTE: Setting to a static renderer here so that the notebook saved size is reduced.
fig_kwargs = {
    # "renderer": "notebook",
    "renderer": "png",
    "width": 1000,
    "height": 600,
}

In [5]:
# 4. Load your Excel data
file_path = '15skulist.xlsx'  # adjust path as needed
df = pd.read_excel(file_path, engine='openpyxl')
df['Months'] = pd.to_datetime(df['Months'])
df = df.sort_values(['Items', 'Months']).reset_index(drop=True)

In [6]:
# 5. Define global forecasting params
fh = 12      # forecast next 12 months
fold = 3     # 3-fold CV
fig_kwargs = {"renderer": "png", "width": 1000, "height": 600}


In [1]:
import dask
import sktime

print(f"dask version: {dask.__version__}")
print(f"sktime version: {sktime.__version__}")

/usr/lib/python3.11/importlib/__init__.py:126: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  return _bootstrap._gcd_import(name[level:], package, level)


dask version: 2024.12.1
sktime version: 0.26.0


In [9]:
!pip install "dask<2025.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.4 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2025.5.0
    Uninstalling dask-2025.5.0:
      Successfully uninstalled dask-2025.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rapids-dask-dependency 25.6.0 requires dask==2025.5.0, but you have dask 2024.12.1 which is incompatible.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.
distributed 2025.5.0 requires dask==2025.5.0, but you have dask 2024.12.1 which is incompatible.


In [7]:
import matplotlib.pyplot as plt

In [8]:
# 6. Prepare container for all forecasts
all_forecasts = []

# 7. Loop over each SKU
for sku in df['Items'].unique():
    sku_df = df[df['Items'] == sku].copy()
    sku_df = sku_df.set_index('Months').asfreq('MS')  # enforce monthly freq

    # skip if too short for 3 folds + 12-step horizon
    if len(sku_df) < fold + fh:
        print(f"Skipping {sku}: insufficient data ({len(sku_df)} records)")
        continue

    # 8. Initialize PyCaret experiment
    exp = TSForecastingExperiment()
    exp.setup(
        data=sku_df['Sales'],
        fh=fh,
        fold=fold,
        fig_kwargs=fig_kwargs,
        session_id=42,
        verbose=False,
        numeric_imputation_target='mean'
    )

    # 9. Train, tune, finalize
    best = exp.compare_models(sort='MAE')
    tuned = exp.tune_model(best, optimize='MAE')
    final = exp.finalize_model(tuned)

    # 10. Generate forecast DataFrame
    forecast = exp.predict_model(final, fh=fh).reset_index()
    forecast = forecast.rename(columns={'index': 'Months', 'y_pred': 'Sales'})
    forecast['Items'] = sku
    forecast['Type'] = 'Forecast'

    # 11. Append to master list
    all_forecasts.append(forecast[['Months', 'Items', 'Type', 'Sales']])

    # 12. Plot actual + forecast
    plt.figure(figsize=(10, 6))
    # actual historical
    plt.plot(sku_df.index, sku_df['Sales'], label='Actual', color='blue')
    # forecast
    plt.plot(forecast['Months'], forecast['Sales'], label='Forecast', color='red', linestyle='--')
    plt.axvline(sku_df.index[-1], color='gray', linestyle=':', label='Train/Test Split')
    plt.title(f'Forecast for {sku}', fontsize=16)
    plt.xlabel('Months')
    plt.ylabel('Sales')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'forecast_{sku}.png', dpi=300)
    plt.close()

,Model,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2,TT (Sec)
snaive,Seasonal Naive Forecaster,0.8785,0.8661,2371.9444,2848.4712,0.3414,0.3077,-0.7691,0.0867
grand_means,Grand Means Forecaster,0.9203,0.9057,2493.4358,2982.9084,0.3867,0.3300,-0.9746,0.0533
stlf,STLF,0.9717,0.9226,2620.3889,3030.3888,0.3782,0.3174,-1.2021,0.0733
arima,ARIMA,0.9848,0.9592,2654.8320,3151.6346,0.3717,0.3239,-1.3184,0.1033
croston,Croston,0.9881,0.9504,2667.2214,3134.6858,0.3730,0.3582,-0.8843,0.0433
polytrend,Polynomial Trend Forecaster,1.0497,1.0276,2841.5133,3376.4511,0.4783,0.3572,-1.6950,0.0433
huber_cds_dt,Huber w/ Cond. Deseasonalize & Detrending,1.2341,1.2541,3350.9172,4116.3771,0.5431,0.3957,-3.6817,0.3400
naive,Naive Forecaster,1.2709,1.1613,3420.5556,3832.5424,0.4493,0.4708,-1.6357,1.4600
lightgbm_cds_dt,Light Gradient Boosting w/ Cond. Deseasonalize & Detrending,1.3200,1.2390,3566.5046,4078.8966,0.4982,0.4471,-2.4831,0.8067
br_cds_dt,Bayesian Ridge w/ Cond. Deseasonalize & Detrending,1.3607,1.2647,3675.4920,4163.8263,0.5076,0.4597,-2.6003,0.5067


Processing:   0%|          | 0/117 [00:00<?, ?it/s]

,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,2021-05,1.0088,0.9750,2688.3333,3245.9077,0.3015,0.3336,-0.0157
1,2022-05,1.0897,1.0115,2925.8333,3295.3895,0.4402,0.3599,-1.5753
2,2023-05,0.5370,0.6119,1501.6667,2004.1166,0.2826,0.2295,-0.7165
Mean,NaT,0.8785,0.8661,2371.9444,2848.4712,0.3414,0.3077,-0.7691
SD,NaT,0.2437,0.1804,622.9709,597.3905,0.0703,0.0563,0.6378


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 3 folds for each of 4 candidates, totalling 12 fits


...

Processing:   0%|          | 0/117 [00:00<?, ?it/s]

,cutoff,MASE,RMSSE,MAE,RMSE,MAPE,SMAPE,R2
0,2021-05,0.5889,0.6675,1082.0898,1400.4973,0.1947,0.1766,0.6474
1,2022-05,0.4076,0.4491,1177.5889,1444.4696,0.1293,0.1255,0.4949
2,2023-05,0.4336,0.5428,1322.9194,1835.1706,0.1273,0.1132,0.6251
Mean,NaT,0.4767,0.5531,1194.1994,1560.0459,0.1504,0.1384,0.5891
SD,NaT,0.0800,0.0895,99.0174,195.3691,0.0313,0.0275,0.0672


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 3 folds for each of 10 candidates, totalling 30 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  27 out of  30 | elapsed:    1.9s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    2.0s finished


In [9]:
# 13. Combine all SKU forecasts and export CSV
if all_forecasts:
    result_df = pd.concat(all_forecasts, ignore_index=True)
    result_df.to_csv('15skulist_all_sku_forecasts.csv', index=False)
    print("✅ All SKU forecasts saved to '15skulist_all_sku_forecasts.csv'")
else:
    print("⚠️ No forecasts generated.")

✅ All SKU forecasts saved to '15skulist_all_sku_forecasts.csv'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [10]:
fc=pd.read_csv('/content/15skulist_all_sku_forecasts.csv')

In [11]:
fc

,Months,Items,Type,Sales
0,2025-06,B10003-2.5,Forecast,3980.0000
1,2025-07,B10003-2.5,Forecast,4710.0000
2,2025-08,B10003-2.5,Forecast,5040.0000
3,2025-09,B10003-2.5,Forecast,5070.0000
4,2025-10,B10003-2.5,Forecast,4300.0000
...,...,...,...,...
175,2026-01,B8015-5,Forecast,18750.8132
176,2026-02,B8015-5,Forecast,23174.1316
177,2026-03,B8015-5,Forecast,20792.4010
178,2026-04,B8015-5,Forecast,20196.2155
